# Steam Analytics — Pipeline completo
## Universidad de La Sabana · Analítica Aplicada

In [ ]:
# ============================================================
# LIBRERÍAS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
sns.set_theme(style='darkgrid')


In [ ]:
# ============================================================
# RUTAS — ajusta si tus archivos estan en otra carpeta
# Importante: Python distingue mayusculas en Linux/Mac
#   'games.csv' != 'Games.csv'
# ============================================================
RUTA_GAMES    = 'games.csv'              # <- minuscula
RUTA_REVIEWS  = 'reviews.csv'
RUTA_STEAMSPY = 'steamspy_insights.csv'
RUTA_GENRES   = 'genres.csv'


## PASO 1 — Leer archivos

In [ ]:
def leer_archivos():
    """
    Lee los 4 CSV con engine='python' en todos para evitar el ParserError
    del motor C cuando hay comas dentro de campos entrecomillados.
    """
    print("[1] Leyendo archivos...")

    opts = dict(engine='python', on_bad_lines='skip')

    games_df    = pd.read_csv(RUTA_GAMES,    **opts)
    reviews_df  = pd.read_csv(RUTA_REVIEWS,  **opts)
    steamspy_df = pd.read_csv(RUTA_STEAMSPY, **opts)
    genres_df   = pd.read_csv(RUTA_GENRES,   **opts)

    print(f"   games:    {games_df.shape}  | cols: {list(games_df.columns[:5])}")
    print(f"   reviews:  {reviews_df.shape} | cols: {list(reviews_df.columns[:5])}")
    print(f"   steamspy: {steamspy_df.shape} | cols: {list(steamspy_df.columns[:5])}")
    print(f"   genres:   {genres_df.shape}  | cols: {list(genres_df.columns)}")
    return games_df, reviews_df, steamspy_df, genres_df


## PASO 2 — Limpiar datos nulos

In [ ]:
def limpiar_datos_nulos(games_df, reviews_df, steamspy_df, genres_df):
    """
    Elimina filas donde las columnas CRITICAS tienen NaN.
    Solo tocamos las columnas que usamos directamente en el análisis.
    """
    print("[2] Limpiando nulos...")

    # games: necesitamos app_id y fecha
    col_fecha = 'date_release' if 'date_release' in games_df.columns else 'release_date'
    antes = len(games_df)
    games_df = games_df.dropna(subset=['app_id', col_fecha]).copy()
    print(f"   games:    {antes} -> {len(games_df)}")

    # reviews: positive y negative son los que usamos para approval_rate
    # (tu archivo ya tiene los conteos por juego, no filas individuales)
    antes = len(reviews_df)
    reviews_df = reviews_df.dropna(subset=['app_id', 'positive', 'negative']).copy()
    print(f"   reviews:  {antes} -> {len(reviews_df)}")

    # steamspy: owners_range y concurrent_users_yesterday
    antes = len(steamspy_df)
    steamspy_df = steamspy_df.dropna(subset=['app_id', 'owners_range']).copy()
    print(f"   steamspy: {antes} -> {len(steamspy_df)}")

    return games_df, reviews_df, steamspy_df, genres_df


## PASO 3 — Limpiar datos faltantes + calcular approval_rate

In [ ]:
def limpiar_datos_faltantes(games_df, reviews_df):
    """
    Aplica filtros de calidad y calcula la variable objetivo: approval_rate.

    approval_rate = (positivas / (positivas + negativas)) * 100

    TU archivo reviews.csv ya tiene los conteos agregados por juego
    (columnas 'positive' y 'negative'), asi que calculamos directo —
    no necesitamos agrupar ni contar filas individuales.

    Filtros:
    - Solo juegos (no DLCs ni demos)
    - Fechas validas 2000-2024
    - Al menos 10 reseñas (con menos, el porcentaje no es representativo)
    """
    print("[3] Calculando approval_rate y filtrando...")

    # -- Approval rate directo desde positive/negative --
    reviews_df = reviews_df.copy()
    reviews_df['positive']  = pd.to_numeric(reviews_df['positive'], errors='coerce')
    reviews_df['negative']  = pd.to_numeric(reviews_df['negative'], errors='coerce')
    reviews_df['total_calc'] = reviews_df['positive'] + reviews_df['negative']

    # Filtrar juegos con al menos 10 reseñas
    reviews_df = reviews_df[reviews_df['total_calc'] >= 10].copy()

    # Calcular approval_rate
    reviews_df['approval_rate'] = (reviews_df['positive'] / reviews_df['total_calc']) * 100

    # -- Filtros en games --
    col_fecha = 'date_release' if 'date_release' in games_df.columns else 'release_date'
    games_df = games_df.copy()
    games_df[col_fecha] = pd.to_datetime(games_df[col_fecha], errors='coerce')
    games_df = games_df.dropna(subset=[col_fecha])
    games_df['year'] = games_df[col_fecha].dt.year
    games_df = games_df[
        (games_df['year'] >= 2000) & (games_df['year'] <= 2024)
    ].copy()

    # Filtro tipo juego (si existe la columna)
    if 'app_type' in games_df.columns:
        games_df = games_df[games_df['app_type'] == 'game'].copy()
    elif 'type' in games_df.columns:
        games_df = games_df[games_df['type'] == 'game'].copy()

    # Filtro precio (si existe)
    col_precio = None
    for c in ['price_final', 'price', 'Price']:
        if c in games_df.columns:
            col_precio = c
            break
    if col_precio:
        games_df[col_precio] = pd.to_numeric(games_df[col_precio], errors='coerce')
        games_df = games_df[
            (games_df[col_precio] >= 0) & (games_df[col_precio] <= 20000)
        ].copy()
        games_df.rename(columns={col_precio: 'price_final'}, inplace=True)

    # -- Unir --
    df = pd.merge(games_df, reviews_df[['app_id', 'approval_rate', 'total_calc']], on='app_id', how='inner')
    df.rename(columns={'total_calc': 'total_reviews'}, inplace=True)

    print(f"   Juegos validos: {len(df)}")
    print(f"   approval_rate media: {df['approval_rate'].mean():.1f}%")
    print(f"   approval_rate std:   {df['approval_rate'].std():.1f}%")
    return df


## PASO 4 — Parsear owners_range

In [ ]:
def parsear_owners(steamspy_df):
    """
    'Parsear' = leer un texto y convertirlo en un numero usable.

    SteamSpy guarda propietarios como rango de texto:
        '10,000,000 .. 20,000,000'  -> punto medio = 15,000,000
        '0 .. 20,000'               -> punto medio = 10,000

    Pasos:
    1. Quitar comas de miles: '10,000,000' -> '10000000'
    2. Separar en dos partes con ' .. ' como separador (con espacios)
    3. Convertir a enteros y promediar
    """
    print("[4] Parseando owners_range -> owners_mid...")

    def punto_medio(texto):
        try:
            limpio = str(texto).replace(',', '')   # quitar comas de miles
            partes = limpio.split('..')             # separar en [inf, sup]
            inf = int(partes[0].strip())
            sup = int(partes[1].strip())
            return (inf + sup) / 2
        except Exception:
            return np.nan

    steamspy_df = steamspy_df.copy()
    steamspy_df['owners_mid'] = steamspy_df['owners_range'].apply(punto_medio)
    steamspy_df['concurrent_users_yesterday'] = pd.to_numeric(
        steamspy_df['concurrent_users_yesterday'], errors='coerce'
    )

    validos = steamspy_df['owners_mid'].notna().sum()
    print(f"   owners_mid calculados: {validos}/{len(steamspy_df)}")
    return steamspy_df


## PASO 5 — Fusionar datasets

In [ ]:
def fusionar_datasets(games_df, steamspy_df, genres_df):
    """
    Une todas las fuentes en un DataFrame principal y uno relajado.

    df           -> Requiere approval_rate (para correlaciones)
    df_concurrent-> Solo necesita concurrent_users_yesterday, no approval_rate
                    (así no perdemos juegos con muchos usuarios pero pocas reseñas)
    """
    print("[5] Fusionando datasets...")

    cols_spy = ['app_id', 'owners_mid', 'concurrent_users_yesterday']
    if 'price' in steamspy_df.columns:
        cols_spy.append('price')

    # -- TABLA PRINCIPAL --
    df = pd.merge(games_df, steamspy_df[cols_spy], on='app_id', how='left')

    # Agregar primer género de genres.csv
    if 'genre' in genres_df.columns:
        primer_genero = genres_df.groupby('app_id')['genre'].first().reset_index()
        primer_genero.columns = ['app_id', 'genre']
        df = pd.merge(df, primer_genero, on='app_id', how='left')

    print(f"   df principal: {df.shape}")

    # -- TABLA RELAJADA para concurrent_users --
    col_nombre = 'name' if 'name' in games_df.columns else 'app_id'
    df_concurrent = pd.merge(
        games_df[['app_id', col_nombre, 'year']],
        steamspy_df[cols_spy],
        on='app_id',
        how='inner'   # Solo los que tienen datos SteamSpy
    )
    df_concurrent = pd.merge(
        df_concurrent,
        games_df[['app_id', 'approval_rate']].drop_duplicates() if 'approval_rate' in games_df.columns
        else df[['app_id', 'approval_rate']].drop_duplicates(),
        on='app_id',
        how='left'
    )
    print(f"   df_concurrent: {df_concurrent.shape}")
    return df, df_concurrent


## PASO 6 — Modelo de referencia (null model)

In [ ]:
def modelo_de_referencia(df):
    """
    El modelo nulo predice la media global para todos los juegos.
    Es el piso minimo que cualquier modelo de ML debe superar.
    RMSE del modelo nulo = desviacion estandar de approval_rate (matematicamente).
    """
    print("[6] Calculando modelo de referencia...")
    y_real        = df['approval_rate'].dropna()
    media_global  = y_real.mean()
    y_pred        = np.full(len(y_real), media_global)
    errores       = y_real.values - y_pred
    mae           = np.mean(np.abs(errores))
    rmse          = np.sqrt(np.mean(errores ** 2))

    print(f"   Media global:  {media_global:.1f}%")
    print(f"   MAE  nulo:     {mae:.2f} pp")
    print(f"   RMSE nulo:     {rmse:.2f} pp  (= std = {y_real.std():.2f})")
    print(f"   -> Nuestro modelo debe tener RMSE < {rmse:.2f}")
    return {'media': media_global, 'mae': mae, 'rmse': rmse}


## PASO 7 — Estadísticas descriptivas

In [ ]:
def calcular_estadisticas_descriptivas(df):
    print("[7] Estadisticas descriptivas...")
    cols_num = [c for c in ['approval_rate', 'price_final', 'total_reviews', 'owners_mid'] if c in df.columns]
    print(df[cols_num].describe().round(2))

    por_genero = None
    if 'genre' in df.columns:
        por_genero = (df.groupby('genre')['approval_rate']
                      .agg(['mean','count'])
                      .rename(columns={'mean':'aprobacion_media','count':'n_juegos'})
                      .sort_values('n_juegos', ascending=False).head(10))
        print("\n--- Top 10 géneros ---")
        print(por_genero.round(2))

    por_anio = (df.groupby('year')['approval_rate']
                .agg(['mean','count'])
                .rename(columns={'mean':'aprobacion_media','count':'n_juegos'}))
    print("\n--- Por año ---")
    print(por_anio.round(2))
    return por_genero, por_anio


## PASO 8 — Gráfica: juegos por año

In [ ]:
def graficar_distribucion_juegos(df):
    print("[8] Juegos por año...")
    conteo = df.groupby('year').size().reset_index(name='n_juegos')
    fig, ax = plt.subplots(figsize=(14,5))
    ax.bar(conteo['year'], conteo['n_juegos'], color='#5B8DEF', edgecolor='white', linewidth=0.5)
    ax.set_title('Juegos lanzados en Steam por año', fontsize=16, fontweight='bold')
    ax.set_xlabel('Año'); ax.set_ylabel('Número de juegos')
    plt.tight_layout()
    plt.savefig('fig_juegos_por_anio.png', dpi=150)
    plt.show()


## PASO 9 — Gráfica: distribución approval_rate

In [ ]:
def graficar_distribucion_scores(df):
    print("[9] Distribución de approval_rate...")
    fig, ax = plt.subplots(figsize=(10,5))
    ax.hist(df['approval_rate'], bins=40, color='#00C48C', edgecolor='white', linewidth=0.3)
    media = df['approval_rate'].mean()
    ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Media: {media:.1f}%')
    ax.set_title('Distribución de approval_rate (variable objetivo)', fontsize=16)
    ax.set_xlabel('Porcentaje de aprobación (%)'); ax.set_ylabel('Número de juegos')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig_distribucion_scores.png', dpi=150)
    plt.show()


## PASO 10 — Gráfica: aprobación por género

In [ ]:
def graficar_aprobacion_por_genero(df):
    print("[10] Aprobación por género...")
    if 'genre' not in df.columns:
        print("   Columna genre no disponible.")
        return
    top_generos = df['genre'].value_counts().head(10).index
    df_top = df[df['genre'].isin(top_generos)]
    orden  = df_top.groupby('genre')['approval_rate'].median().sort_values(ascending=False).index
    fig, ax = plt.subplots(figsize=(14,6))
    sns.boxplot(data=df_top, x='genre', y='approval_rate', order=orden, palette='viridis', ax=ax)
    ax.set_title('Distribución de approval_rate por género (Top 10)', fontsize=15)
    ax.set_xlabel('Género'); ax.set_ylabel('approval_rate (%)')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig('fig_aprobacion_genero.png', dpi=150)
    plt.show()


## PASO 11 — Gráfica: top juegos por usuarios concurrentes

In [ ]:
def graficar_top_concurrentes(df_concurrent):
    """
    Usa df_concurrent (tabla relajada) para no perder juegos
    que tienen muchos usuarios pero pocas reseñas en nuestro dataset.
    La columna en tu steamspy es: concurrent_users_yesterday
    """
    print("[11] Top juegos por concurrent_users_yesterday...")
    col_ccu = 'concurrent_users_yesterday'
    if col_ccu not in df_concurrent.columns:
        print(f"   Columna {col_ccu} no disponible.")
        return

    col_nombre = 'name' if 'name' in df_concurrent.columns else 'app_id'
    top = (df_concurrent[df_concurrent[col_ccu] > 0]
           .sort_values(col_ccu, ascending=False).head(20))

    fig, ax = plt.subplots(figsize=(12,8))
    ax.barh(top[col_nombre].astype(str), top[col_ccu] / 1_000,
            color='#FF6B6B', edgecolor='white', linewidth=0.4)
    ax.set_title('Top 20 juegos por usuarios concurrentes (ayer)', fontsize=15)
    ax.set_xlabel('Usuarios concurrentes (miles)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('fig_top_concurrentes.png', dpi=150)
    plt.show()


## PASO 12 — Correlaciones de Pearson

In [ ]:
def calcular_correlaciones_pearson(df):
    """
    r de Pearson mide fuerza y dirección de relación LINEAL entre dos variables.
        r =  1.0  -> relación positiva perfecta
        r =  0.0  -> sin relación
        r = -1.0  -> relación negativa perfecta
    p-value < 0.05 = estadísticamente significativo (no es por azar).
    """
    print("[12] Correlaciones de Pearson...")
    candidatos = ['price_final', 'total_reviews', 'owners_mid', 'concurrent_users_yesterday']
    predictores = [c for c in candidatos if c in df.columns]

    resultados = []
    for p in predictores:
        temp = df[[p, 'approval_rate']].dropna()
        if len(temp) < 30: continue
        r, pval = stats.pearsonr(temp[p], temp['approval_rate'])
        resultados.append({
            'predictor': p, 'r_pearson': round(r,4),
            'p_value': round(pval,6), 'n': len(temp),
            'significativo': 'Si' if pval < 0.05 else 'No'
        })

    tabla = pd.DataFrame(resultados).sort_values('r_pearson', key=abs, ascending=False)
    print("\n--- Tabla de correlaciones ---")
    print(tabla.to_string(index=False))
    return tabla


## PASO 13 — Panel de scatter + regresión

In [ ]:
def graficar_panel_correlaciones(df, tabla_corr):
    print("[13] Panel scatter + líneas de regresión...")
    predictores = tabla_corr['predictor'].tolist()
    n = len(predictores)
    if n == 0:
        print("   Sin predictores."); return

    cols = 2
    rows = (n + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5*rows))
    axes = np.array(axes).flatten()

    for i, pred in enumerate(predictores):
        temp = df[[pred, 'approval_rate']].dropna()
        r    = tabla_corr.loc[tabla_corr['predictor']==pred, 'r_pearson'].values[0]
        axes[i].scatter(temp[pred], temp['approval_rate'], alpha=0.3, s=8, color='#5B8DEF')
        m, b = np.polyfit(temp[pred], temp['approval_rate'], 1)
        x_l  = np.linspace(temp[pred].min(), temp[pred].max(), 100)
        axes[i].plot(x_l, m*x_l+b, color='red', linewidth=2)
        axes[i].set_title(f'{pred} vs approval_rate  (r = {r})', fontsize=11)
        axes[i].set_xlabel(pred); axes[i].set_ylabel('approval_rate (%)')

    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    plt.suptitle('Panel: predictores vs approval_rate', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.savefig('fig_panel_correlaciones.png', dpi=150, bbox_inches='tight')
    plt.show()


## PASO 14 — Heatmap género × precio (pantalla principal)

In [ ]:
def pantalla_principal_usuario(df):
    """
    La pregunta de negocio: ¿qué combinación género + rango de precio
    tiene mayor aprobación?  Respuesta para un desarrollador indie.
    """
    print("[14] Heatmap género x precio...")
    if 'genre' not in df.columns or 'price_final' not in df.columns:
        print("   Faltan columnas genre o price_final.")
        return

    df = df.copy()
    bins   = [-0.01, 0, 500, 1500, 3000, 20000]
    labels = ['Gratis', '$0-5', '$5-15', '$15-30', '$30+']
    df['rango_precio'] = pd.cut(df['price_final'], bins=bins, labels=labels)

    top_g = df['genre'].value_counts().head(12).index
    pivot = (df[df['genre'].isin(top_g)]
             .pivot_table(values='approval_rate', index='genre',
                          columns='rango_precio', aggfunc='mean'))

    fig, ax = plt.subplots(figsize=(12,8))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
                linewidths=0.5, ax=ax, cbar_kws={'label':'approval_rate media (%)'})
    ax.set_title('approval_rate media por Género × Precio\n(verde = más aprobado)',
                 fontsize=15, fontweight='bold')
    ax.set_xlabel('Rango de precio (USD)'); ax.set_ylabel('Género')
    plt.tight_layout()
    plt.savefig('fig_heatmap.png', dpi=150)
    plt.show()


## MAIN — Orquesta todo el pipeline

In [ ]:
def main():
    print("=" * 60)
    print("  STEAM ANALYTICS — Pipeline completo")
    print("=" * 60)

    # PASO 1: Leer CSV en crudo
    games_df, reviews_df, steamspy_df, genres_df = leer_archivos()

    # PASO 2: Eliminar NaN en columnas críticas
    games_df, reviews_df, steamspy_df, genres_df = limpiar_datos_nulos(
        games_df, reviews_df, steamspy_df, genres_df
    )

    # PASO 3: Filtrar + calcular approval_rate desde positive/negative
    games_df = limpiar_datos_faltantes(games_df, reviews_df)

    # PASO 4: Convertir 'N,NNN,NNN .. M,MMM,MMM' a número (punto medio)
    steamspy_df = parsear_owners(steamspy_df)

    # PASO 5: Unir todo -> df principal + df_concurrent (relajado)
    df, df_concurrent = fusionar_datasets(games_df, steamspy_df, genres_df)

    # PASO 6: Baseline nulo (piso que debemos superar)
    referencia = modelo_de_referencia(df)

    # PASO 7: Estadísticas descriptivas
    calcular_estadisticas_descriptivas(df)

    # PASO 8: Juegos por año
    graficar_distribucion_juegos(df)

    # PASO 9: Distribución de approval_rate
    graficar_distribucion_scores(df)

    # PASO 10: Boxplot por género
    graficar_aprobacion_por_genero(df)

    # PASO 11: Top concurrentes (usa tabla relajada)
    graficar_top_concurrentes(df_concurrent)

    # PASO 12: Correlaciones Pearson
    tabla_corr = calcular_correlaciones_pearson(df)

    # PASO 13: Scatter panel con líneas de regresión
    graficar_panel_correlaciones(df, tabla_corr)

    # PASO 14: Heatmap género × precio (respuesta al desarrollador indie)
    pantalla_principal_usuario(df)

    print("\n" + "=" * 60)
    print("  Pipeline completo.")
    print(f"  Modelo nulo RMSE: {referencia['rmse']:.2f} pp")
    print("=" * 60)
    return df, df_concurrent, tabla_corr, referencia


df, df_concurrent, tabla_corr, referencia = main()
